In [1]:
# import dependencies
%matplotlib inline
import os
import sys
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from ark.utils.plot_utils import cohort_cluster_plot

# set a working directory
wdir = ('/Users/shihongwu/pancreatic_image_analysis/34434_1/')
os.chdir(wdir)

# Load data

In [2]:
adata_obs_component = pd.read_csv('tmp_phenotype_boundary_of_interest_clustered_epithelial cells.txt')
cluster_of_interest = np.load('clusters_data/regions_to_get_boundaries.npy')

In [3]:
fovs = adata_obs_component["fov"].unique().tolist()

In [4]:
cell_data = adata_obs_component

cell_data_filtered = adata_obs_component[adata_obs_component["component"].isin(cluster_of_interest)]

In [5]:
# Define paths
seg_dir = "ark_wdir/segmentation/deepcell_output/"
save_dir = "epi_cluster"

In [6]:
adata_obs_component

,label,area,eccentricity,major_axis_length,minor_axis_length,perimeter,convex_area,equivalent_diameter,orientation,solidity,...,flag_CD27,flag_CD38,flag_IgA,flag_HLADR,flag_CD11b,cell_ids,simplified_phenotype,phenotype_boundary_of_interest_column,id,component
0,1,34.0,0.612372,7.388385,5.841031,19.656854,38.0,6.579525,1.570796,0.894737,...,Unknown,Unknown,Unknown,Unknown,Unknown,fov0_1,other cells,other cells,fov0_1,singleton
1,2,146.0,0.836650,18.415837,10.087055,46.384776,157.0,13.634257,1.522461,0.929936,...,Unknown,Unknown,IgA+,Unknown,Unknown,fov0_2,other cells,other cells,fov0_2,singleton
2,3,58.0,0.667875,10.076281,7.499509,27.071068,62.0,8.593480,-1.217645,0.935484,...,Unknown,Unknown,IgA+,Unknown,Unknown,fov0_3,other cells,other cells,fov0_3,singleton
3,4,293.0,0.842358,26.363566,14.207810,66.627417,306.0,19.314740,1.567566,0.957516,...,Unknown,Unknown,IgA+,Unknown,Unknown,fov0_4,other cells,other cells,fov0_4,singleton
4,5,58.0,0.777649,10.821477,6.803442,26.727922,62.0,8.593480,1.492026,0.935484,...,Unknown,Unknown,IgA+,Unknown,Unknown,fov0_5,other cells,other cells,fov0_5,singleton
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1766476,187194,146.0,0.933493,22.744546,8.156102,50.970563,153.0,13.634257,1.551653,0.954248,...,Unknown,Unknown,Unknown,Unknown,Unknown,fov5_187194,other cells,other cells,fov5_187194,singleton
1766477,187195,286.0,0.918906,30.410885,11.996350,71.455844,303.0,19.082623,-1.562425,0.943894,...,Unknown,Unknown,Unknown,Unknown,Unknown,fov5_187195,other cells,other cells,fov5_187195,singleton
1766478,187196,184.0,0.895768,22.965008,10.208470,53.798990,191.0,15.306080,1.559143,0.963351,...,Unknown,Unknown,Unknown,Unknown,Unknown,fov5_187196,other cells,other cells,fov5_187196,singleton
1766479,187197,93.0,0.898412,16.410161,7.206593,37.313708,97.0,10.881695,-1.566779,0.958763,...,Unknown,Unknown,Unknown,Unknown,Unknown,fov5_187197,other cells,other cells,fov5_187197,singleton


# proliferating index (Ki67 +) generation

In [7]:
from tqdm import tqdm

def calculate_proliferating_index(cell_data: pd.DataFrame, cluster_of_interest: list) -> pd.DataFrame:
    """
    Calculate the proliferating index for each cluster based on Ki67+ expression.

    Args:
        cell_data (pd.DataFrame): DataFrame containing cell-level information (e.g., component, flag_Ki67).
        cluster_of_interest (list): List of cluster IDs of interest.

    Returns:
        pd.DataFrame: DataFrame containing the proliferating index for each cluster.
    """
    proliferating_data = []

    for cluster_id in tqdm(cluster_of_interest, desc="Processing Clusters", unit="cluster"):
        # Step 1: Filter cells belonging to the current cluster
        cluster_cells = cell_data[cell_data["component"] == cluster_id]

        if cluster_cells.empty:
            continue  # Skip empty clusters

        # Step 2: Calculate proliferating index based on Ki67+
        total_cells = len(cluster_cells)
        proliferating_cells = cluster_cells[cluster_cells["flag_Ki67"] == "Ki67+"]
        proliferating_count = len(proliferating_cells)

        proliferating_index = (proliferating_count / total_cells) * 100 if total_cells > 0 else 0

        # Step 3: Append data
        proliferating_data.append({
            "cluster_id": cluster_id,
            "total_cells": total_cells,
            "proliferating_cells": proliferating_count,
            "proliferating_index": proliferating_index
        })

    # Convert to DataFrame
    return pd.DataFrame(proliferating_data)

In [8]:
# Example Usage:
# Assuming `cell_data` is your DataFrame and `cluster_of_interest` contains cluster IDs
proliferating_index_df = calculate_proliferating_index(cell_data, cluster_of_interest)

Processing Clusters: 100%|█████████████| 9903/9903 [06:02<00:00, 27.36cluster/s]


# cluster thickness calculation (v3)

In [9]:
from joblib import Parallel, delayed
import numpy as np
from skimage.morphology import medial_axis, remove_small_holes, binary_closing
from skimage.io import imread
from skimage.transform import rescale
import os
import pandas as pd
from tqdm import tqdm  # Import tqdm for progress tracking

# ** Downscaling Factor ** (Set to 0.5 for halving resolution)
DOWNSCALE_FACTOR = 0.5  

def process_single_cluster(cluster_id, cell_data, segmentation_masks, hole_threshold=20):
    """
    Process a single cluster to calculate thickness statistics.
    """
    try:
        # Identify the FOV for this cluster
        cluster_fov = cell_data[cell_data["component"] == cluster_id]["fov"].iloc[0]

        # Ensure the FOV is in the segmentation masks
        if cluster_fov not in segmentation_masks:
            return None  # Skip if FOV not found

        # Get the segmentation mask for this FOV
        segmentation_labels = segmentation_masks[cluster_fov]

        # Extract cell labels for the current cluster
        cluster_cells = cell_data[cell_data["component"] == cluster_id]["label"]

        # Create a binary mask for the current cluster
        binary_mask = np.isin(segmentation_labels, cluster_cells)

        # Fill small holes and refine the mask
        refined_mask = binary_closing(remove_small_holes(binary_mask, area_threshold=hole_threshold))

        # Compute the medial axis and distance transform
        skeleton, distance_map = medial_axis(refined_mask, return_distance=True)

        # Scale thickness values back to original resolution
        medial_thickness = (distance_map[skeleton] * 2) / DOWNSCALE_FACTOR
        
        # Calculate thickness statistics
        return {
            "fov": cluster_fov,
            "cluster_id": cluster_id,
            "max_thickness": np.max(medial_thickness) if medial_thickness.size > 0 else np.nan,
            "mean_thickness": np.mean(medial_thickness) if medial_thickness.size > 0 else np.nan,
            "median_thickness": np.median(medial_thickness) if medial_thickness.size > 0 else np.nan,
            "std_thickness": np.std(medial_thickness) if medial_thickness.size > 0 else np.nan,
        }

    except Exception as e:
        print(f"Error processing cluster {cluster_id}: {e}")
        return None


def calculate_cluster_thickness_parallel(cell_data, cluster_of_interest, seg_dir, fovs, hole_threshold=20, n_jobs=10):
    """
    Parallelized function to calculate thickness statistics for tumor clusters.

    Args:
        cell_data (pd.DataFrame): Cell-level data.
        cluster_of_interest (list): List of cluster IDs to process.
        seg_dir (str): Directory containing segmentation mask files.
        fovs (list): List of FOVs.
        hole_threshold (int): Minimum hole area to fill.
        n_jobs (int): Number of CPU cores to use.

    Returns:
        pd.DataFrame: Thickness statistics for each cluster.
    """
    print("[INFO] Loading and downscaling segmentation masks into memory...")

    # ** Load and Downscale Segmentation Masks **
    segmentation_masks = {
        fov: rescale(imread(os.path.join(seg_dir, f"{fov}_whole_cell.tiff")).astype(int), 
                     DOWNSCALE_FACTOR, anti_aliasing=False, preserve_range=True).astype(int)
        for fov in tqdm(fovs, desc="Loading FOVs", unit="FOV")
    }
    print("[✅] Finished loading & downscaling segmentation masks.")

    # Parallel execution using Joblib
    print("[INFO] Processing clusters in parallel...")
    results = Parallel(n_jobs=n_jobs, backend="loky")(
        delayed(process_single_cluster)(cluster_id, cell_data, segmentation_masks, hole_threshold)
        for cluster_id in tqdm(cluster_of_interest, desc="Processing Clusters", unit="cluster")
    )

    # Remove None results (errors or skipped FOVs)
    thickness_data = [res for res in results if res is not None]

    return pd.DataFrame(thickness_data)

In [10]:
# Example Usage
thickness_stats = calculate_cluster_thickness_parallel(cell_data, 
                                                       cluster_of_interest, 
                                                       seg_dir, 
                                                       fovs, 
                                                       hole_threshold=20, 
                                                       n_jobs=10)

[INFO] Loading and downscaling segmentation masks into memory...


Loading FOVs: 100%|██████████████████████████████| 6/6 [00:14<00:00,  2.35s/FOV]


[✅] Finished loading & downscaling segmentation masks.
[INFO] Processing clusters in parallel...


/Users/shihongwu/anaconda3/envs/ark_env/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
Processing Clusters: 100%|██████████| 9903/9903 [11:12:26<00:00,  4.07s/cluster]


# merge proliferation index and thickness stats

In [11]:
import pandas as pd

# Merge DataFrames by "cluster_id"
merged_df = proliferating_index_df.merge(
    thickness_stats, 
    on="cluster_id",  # Common column to merge on
    how="inner"       # Use 'inner' join to match rows with the same cluster_id
)

# Display the merged DataFrame
merged_df

,cluster_id,total_cells,proliferating_cells,proliferating_index,fov,max_thickness,mean_thickness,median_thickness,std_thickness
0,34434_1_epithelial cells26135,4455,531,11.919192,fov5,354.581443,30.805935,4.000000,49.026416
1,34434_1_epithelial cells6442,3933,66,1.678108,fov1,122.376468,20.476295,4.000000,22.844819
2,34434_1_epithelial cells12370,3072,17,0.553385,fov3,148.808602,11.957224,4.000000,19.382561
3,34434_1_epithelial cells5871,2836,108,3.808181,fov1,120.598507,13.091732,4.000000,17.672350
4,34434_1_epithelial cells9810,2598,912,35.103926,fov2,158.644256,17.153284,4.000000,22.796492
...,...,...,...,...,...,...,...,...,...
9898,34434_1_epithelial cells2807,5,0,0.000000,fov0,39.597980,14.768911,8.944272,11.496136
9899,34434_1_epithelial cells2773,5,0,0.000000,fov0,41.182521,15.677641,12.649111,12.646510
9900,34434_1_epithelial cells12384,5,0,0.000000,fov3,28.844410,6.386128,4.000000,6.019782
9901,34434_1_epithelial cells18425,5,0,0.000000,fov3,52.153619,19.502439,4.000000,17.697269


In [12]:
# Save results to a CSV
merged_df.to_csv("merged_df.csv", index=False)

# Read the CSV file in 

In [7]:
# Read the CSV file
merged_df = pd.read_csv("merged_df.csv")

In [8]:
merged_df

,cluster_id,total_cells,proliferating_cells,proliferating_index,fov,max_thickness,mean_thickness,median_thickness,std_thickness
0,34434_1_epithelial cells4294,4927,176,3.572153,fov1,132.000000,13.423161,4.0,18.870429
1,34434_1_epithelial cells7436,4491,1413,31.462926,fov2,181.416648,15.767311,4.0,22.199232
2,34434_1_epithelial cells19995,4459,531,11.908500,fov5,354.581443,30.701886,4.0,48.856531
3,34434_1_epithelial cells4892,4098,66,1.610542,fov1,122.376468,20.382533,4.0,22.626275
4,34434_1_epithelial cells9366,3200,18,0.562500,fov3,148.808602,11.837178,4.0,19.107040
...,...,...,...,...,...,...,...,...,...
8272,34434_1_epithelial cells5658,5,0,0.000000,fov1,40.792156,20.187267,24.0,11.977722
8273,34434_1_epithelial cells20149,5,1,20.000000,fov5,56.850682,14.651048,4.0,16.927766
8274,34434_1_epithelial cells8754,5,0,0.000000,fov2,43.266615,12.361457,4.0,11.938080
8275,34434_1_epithelial cells1933,5,0,0.000000,fov0,39.395431,12.216453,4.0,11.314656


# Basic cluster morphology calculation including clusters and holes (v3)

In [13]:
from skimage.measure import regionprops, label
from skimage.segmentation import find_boundaries
from skimage.io import imread
from scipy.spatial import ConvexHull
from scipy.ndimage import binary_fill_holes
from joblib import Parallel, delayed
from tqdm import tqdm  # Progress bar


def process_single_cluster(cluster_id, cell_data, segmentation_masks, min_hole_area=100):
    """
    Process a single cluster to compute morphology features.
    """
    try:
        # Step 1: Extract FOV for the current cluster
        fov = cell_data[cell_data["component"] == cluster_id]["fov"].iloc[0]

        # Ensure the FOV is in the segmentation masks
        if fov not in segmentation_masks:
            return None, None  # Skip if FOV not found

        segmentation_labels = segmentation_masks[fov]

        # Step 2: Extract cells belonging to the current cluster
        cluster_cells = cell_data[cell_data["component"] == cluster_id]
        cell_labels = cluster_cells["label"].values

        # Create a binary mask for the cluster
        binary_mask = np.isin(segmentation_labels, cell_labels)

        # Label the cluster mask
        labeled_mask = label(binary_mask)
        cluster_props = regionprops(labeled_mask)

        cluster_features = []
        hole_features = []

        for region in cluster_props:
            # Step 3: Compute cluster and gland features
            centroid_x, centroid_y = region.centroid
            area = region.area
            perimeter = region.perimeter
            eccentricity = region.eccentricity
            convex_area = region.convex_area
            convex_perimeter = ConvexHull(region.coords).area if region.coords.shape[0] >= 3 else np.nan
            convexity = convex_perimeter / perimeter if perimeter > 0 else np.nan

            boundary = find_boundaries(region.image, mode="outer")
            boundary_pixels = np.sum(boundary)
            fractal_dimension = np.log10(boundary_pixels) / np.log10(region.area) if region.area > 0 else np.nan

            compactness = (4 * np.pi * area) / (perimeter ** 2) if perimeter > 0 else np.nan
            elongation = region.major_axis_length / region.minor_axis_length if region.minor_axis_length > 0 else np.nan
            circularity = (4 * np.pi * area) / (perimeter ** 2) if perimeter > 0 else np.nan
            orientation = region.orientation

            # Compute spatial distribution
            gland_cells = cluster_cells.copy()
            gland_cells["distance_to_center"] = np.sqrt(
                (gland_cells["X_centroid"] - centroid_x) ** 2 +
                (gland_cells["Y_centroid"] - centroid_y) ** 2
            )
            bins = np.linspace(0, gland_cells["distance_to_center"].max(), num=10)
            gland_cells["layer"] = np.digitize(gland_cells["distance_to_center"], bins)
            layer_density = gland_cells.groupby("layer").size()
            layer_thickness = np.diff(bins)

            cluster_features.append({
                "fov": fov,
                "cluster_id": cluster_id,
                "region_label": region.label,
                "centroid_x": centroid_x,
                "centroid_y": centroid_y,
                "area": area,
                "perimeter": perimeter,
                "eccentricity": eccentricity,
                "convexity": convexity,
                "fractal_dimension": fractal_dimension,
                "compactness": compactness,
                "elongation": elongation,
                "circularity": circularity,
                "orientation": orientation,
                "mean_radial_deviation": gland_cells["distance_to_center"].mean(),
                "layer_density_variance": layer_density.var() if len(layer_density) > 1 else 0,
                "layer_thickness_variance": layer_thickness.var() if len(layer_thickness) > 1 else 0,
                "spatial_entropy": -np.sum(
                    (layer_density / len(gland_cells)) * np.log(layer_density / len(gland_cells))
                    if len(layer_density) > 0 else 0
                )
            })

        # Step 4: Detect and analyze holes
        filled_mask = binary_fill_holes(binary_mask)
        holes_mask = filled_mask & ~binary_mask
        labeled_holes = label(holes_mask)
        holes_props = regionprops(labeled_holes)

        for hole in holes_props:
            if hole.area < min_hole_area:
                continue  # Filter out small holes

            hole_boundary = find_boundaries(hole.image, mode="outer")
            hole_boundary_pixels = np.sum(hole_boundary)
            hole_fractal_dimension = np.log10(hole_boundary_pixels) / np.log10(hole.area) if hole.area > 0 else np.nan
            hole_circularity = (4 * np.pi * hole.area) / (hole.perimeter ** 2) if hole.perimeter > 0 else np.nan

            hole_features.append({
                "fov": fov,
                "cluster_id": cluster_id,
                "hole_label": hole.label,
                "hole_area": hole.area,
                "hole_perimeter": hole.perimeter,
                "hole_eccentricity": hole.eccentricity,
                "hole_circularity": hole_circularity,
                "hole_fractal_dimension": hole_fractal_dimension,
                "hole_orientation": hole.orientation if hasattr(hole, "orientation") else np.nan,
            })

        return cluster_features, hole_features

    except Exception as e:
        print(f"Error processing cluster {cluster_id}: {e}")
        return None, None


def analyze_clusters_and_glands_basic_morphology_parallel(
    cell_data, cluster_of_interest, seg_dir, min_hole_area=100, n_jobs=10
):
    """
    Parallelized function to analyze clusters, glands, and holes.

    Args:
        cell_data (pd.DataFrame): Cell-level data.
        cluster_of_interest (list): List of cluster IDs to process.
        seg_dir (str): Directory containing segmentation masks.
        min_hole_area (int): Minimum area to consider a valid hole.
        n_jobs (int): Number of CPU cores to use.

    Returns:
        pd.DataFrame: Cluster morphology features.
        pd.DataFrame: Hole features.
    """
    # Load segmentation masks into memory
    unique_fovs = cell_data["fov"].unique()
    segmentation_masks = {
        fov: imread(os.path.join(seg_dir, f"{fov}_whole_cell.tiff")).astype(int)
        for fov in unique_fovs
    }

    # Parallel execution using Joblib
    results = Parallel(n_jobs=n_jobs, backend="loky")(
        delayed(process_single_cluster)(cluster_id, cell_data, segmentation_masks, min_hole_area)
        for cluster_id in tqdm(cluster_of_interest, desc="Processing Clusters", unit="cluster")
    )

    # Separate results
    cluster_results, hole_results = zip(*results)

    # Remove None values
    cluster_features = [res for res in cluster_results if res is not None]
    hole_features = [res for res in hole_results if res is not None]

    # Convert to DataFrames
    combined_features_df = pd.DataFrame([item for sublist in cluster_features for item in sublist])
    hole_features_df = pd.DataFrame([item for sublist in hole_features for item in sublist])

    return combined_features_df, hole_features_df

In [14]:
# Example Usage
combined_cluster_features, hole_features = analyze_clusters_and_glands_basic_morphology_parallel(cell_data, 
                                                                                                 cluster_of_interest, 
                                                                                                 seg_dir, 
                                                                                                 min_hole_area=100, 
                                                                                                 n_jobs=10)

/Users/shihongwu/anaconda3/envs/ark_env/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
Processing Clusters:   4%|▎        | 410/9903 [1:02:50<19:25:56,  7.37s/cluster]

Error processing cluster 34434_1_epithelial cells12152: QH6154 Qhull precision error: Initial simplex is flat (facet 1 is coplanar with the interior point)

While executing:  | qhull i Qt
Options selected for Qhull 2019.1.r 2019/06/21:
  run-id 1327461775  incidence  Qtriangulate  _pre-merge  _zero-centrum
  _max-width 14  Error-roundoff 7.8e-12  _one-merge 3.9e-11  _near-inside 2e-10
  Visible-distance 1.6e-11  U-max-coplanar 1.6e-11  Width-outside 3.1e-11
  _wide-facet 9.4e-11  _maxoutside 4.7e-11

The input to qhull appears to be less than 2 dimensional, or a
computation has overflowed.

Qhull could not construct a clearly convex simplex from points:
- p1(v3): 5.9e+02 1.1e+04
- p14(v2): 6e+02 1.1e+04
- p0(v1): 5.9e+02 1.1e+04

The center point is coplanar with a facet, or a vertex is coplanar
with a neighboring facet.  The maximum round off error for
computing distances is 7.8e-12.  The center point, facets and distances
to the center point are as follows:

center point      592 1.1

Processing Clusters: 100%|██████████| 9903/9903 [13:19:53<00:00,  4.85s/cluster]


In [15]:
# Save results to a CSV
combined_cluster_features.to_csv("combined_cluster_features.csv", index=False)
hole_features.to_csv("hole_features.csv", index=False)

# Read the CSV file in

In [9]:
combined_cluster_features = pd.read_csv("combined_cluster_features.csv")
hole_features = pd.read_csv("hole_features.csv")

In [10]:
combined_cluster_features

,fov,cluster_id,region_label,centroid_x,centroid_y,area,perimeter,eccentricity,convexity,fractal_dimension,compactness,elongation,circularity,orientation,mean_radial_deviation,layer_density_variance,layer_thickness_variance,spatial_entropy
0,fov1,34434_1_epithelial cells4294,1,10374.268697,11407.334848,872317,41417.563825,0.974342,0.283688,0.763593,0.006390,4.442977,0.006390,-0.710983,25164.664370,4.196106e+06,3.446586e-25,0.503016
1,fov1,34434_1_epithelial cells4294,2,8884.749617,13436.627871,1306,181.539105,0.844940,0.838082,0.691712,0.497981,1.869644,0.497981,0.460858,26774.235011,4.863077e+06,1.263748e-24,0.437734
2,fov1,34434_1_epithelial cells4294,3,9137.020101,13077.020101,199,64.526912,0.915801,0.904465,0.735234,0.600594,2.489839,0.600594,0.988129,26490.324217,4.680642e+06,1.102907e-24,0.455974
3,fov1,34434_1_epithelial cells4294,4,9154.909091,12643.890909,55,25.071068,0.674692,0.985789,0.485587,1.099580,1.354831,1.099580,-1.342931,26439.081356,4.532706e+06,1.102907e-24,0.470545
4,fov1,34434_1_epithelial cells4294,5,9171.477231,11863.750455,549,87.840620,0.734993,0.957830,0.654260,0.894111,1.474763,0.894111,-1.366535,26380.149291,4.376706e+06,1.378634e-24,0.485708
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37606,fov2,34434_1_epithelial cells8754,2,17740.454713,21656.149723,1082,125.396970,0.748794,0.959483,0.637559,0.864695,1.508744,0.864695,-0.047319,16793.772071,4.500000e+00,3.446586e-25,0.500402
37607,fov2,34434_1_epithelial cells8754,3,17761.045553,21622.173536,461,79.112698,0.668498,0.968390,0.613233,0.925589,1.344602,0.925589,1.364952,16822.083568,4.500000e+00,3.446586e-25,0.500402
37608,fov0,34434_1_epithelial cells1933,1,15644.385738,16587.368823,3015,256.107648,0.672022,0.832565,0.659464,0.577633,1.350382,0.577633,1.445691,1344.363602,4.500000e+00,1.346323e-27,0.500402
37609,fov0,34434_1_epithelial cells1933,2,15681.081871,16655.029240,171,47.798990,0.775411,0.970184,0.601176,0.940521,1.583640,0.940521,0.405651,1367.360443,4.500000e+00,4.936516e-27,0.500402


# aggregate individual cell's morphology features into cluster 

In [16]:
# Select relevant morphology columns from `adata_obs_component`
morphology_columns = [
    "area", "major_axis_length", "minor_axis_length", "perimeter",
    "convex_area", "equivalent_diameter", "orientation", "solidity","feret_diameter_max", 
    "major_minor_axis_ratio", "perim_square_over_area",
    "major_axis_equiv_diam_ratio", "convex_hull_resid", "centroid_dif",
    "num_concavities", "circularity", #"fractal_dimension",
    "boundary_irregularity", 
    "nc_ratio"
]

In [17]:
# **Filter data to include only clusters in `cluster_of_interest`**
filtered_adata_obs_component = adata_obs_component[adata_obs_component["component"].isin(cluster_of_interest)]

In [18]:
# Group by tumor cluster and compute summary statistics
morphology_aggregated = filtered_adata_obs_component.groupby("component")[morphology_columns].agg(
    ["mean", "std", "min", "max", "median"]
).reset_index()
morphology_aggregated

component         area                             \
                                            mean         std    min     max   
0         34434_1_epithelial cells1   971.375000  318.488366  534.0  1413.0   
1        34434_1_epithelial cells10   826.000000  358.949393  292.0  1595.0   
2     34434_1_epithelial cells10001   786.166667  478.548813  241.0  1382.0   
3     34434_1_epithelial cells10003  1195.600000  333.186584  743.0  1504.0   
4     34434_1_epithelial cells10007   744.561404  330.279320  104.0  1579.0   
...                             ...          ...         ...    ...     ...   
9898   34434_1_epithelial cells9986   903.803030  380.803128  161.0  1914.0   
9899    34434_1_epithelial cells999   599.379310  266.233556   62.0  1163.0   
9900   34434_1_epithelial cells9990   538.800000  188.936674  281.0   786.0   
9901   34434_1_epithelial cells9998   741.250000  343.486341  150.0  1416.0   
9902   34434_1_epithelial cells9999  1115.873016  643.709825  139.0  3158.0   

             major_axis_length                                   ...  \
      median              mean        std        min        max  ...   
0      951.5         41.787551   6.344545  31.413721  49.655243  ...   
1      752.0         40.786004   8.423006  23.397096  54.698792  ...   
2      757.0         35.927149  11.203262  18.621426  46.681448  ...   
3     1260.0         51.801055  10.002341  39.213145  62.580616  ...   
4      762.0         36.972646   9.704207  12.360754  52.931312  ...   
...      ...               ...        ...        ...        ...  ...   
9898   916.5         40.106351   9.989119  14.721827  66.924745  ...   
9899   610.0         32.508544   8.544402  10.500690  50.603095  ...   
9900   563.0         31.103496   5.532347  20.595263  38.468956  ...   
9901   649.5         35.328093  10.443240  18.882569  60.947698  ...   
9902  1006.0         45.997449  15.459936  14.512719  88.938489  ...   

     boundary_irregularity                                          nc_ratio  \
                      mean       std       min       max    median      mean   
0                 3.782436  0.084233  3.655584  3.909831  3.783863  0.835448   
1                 3.878581  0.163454  3.641400  4.324321  3.886122  0.905092   
2                 3.765168  0.102978  3.620237  3.911648  3.758217  0.702414   
3                 3.941376  0.149990  3.787041  4.159087  3.953550  0.573629   
4                 3.834246  0.179848  3.390165  4.284072  3.845899  0.776071   
...                    ...       ...       ...       ...       ...       ...   
9898              3.820733  0.135720  3.434597  4.170895  3.812780  0.743187   
9899              3.828370  0.143415  3.553670  4.288858  3.798366  0.914635   
9900              3.859287  0.177555  3.652505  4.201265  3.812508  0.881631   
9901              3.726075  0.118116  3.569389  4.012372  3.701379  0.780571   
9902              3.951432  0.209760  3.588661  4.613515  3.923957  0.638859   

                                              
           std       min       max    median  
0     0.056771  0.726772  0.898797  0.843821  
1     0.477598  0.424908  2.807018  0.819196  
2     0.244669  0.421569  1.136929  0.647711  
3     0.183626  0.317969  0.788501  0.548413  
4     0.370054  0.211538  3.195946  0.771930  
...        ...       ...       ...       ...  
9898  0.530343  0.161491  4.805714  0.703723  
9899  0.555922  0.323300  5.672897  0.815217  
9900  0.452378  0.395018  1.878345  0.733791  
9901  0.448772  0.514706  2.413333  0.690387  
9902  0.269349  0.000000  1.459627  0.696375  

[9903 rows x 91 columns]

In [19]:
# Flatten the column names
morphology_aggregated.columns = ["_".join(col).strip("_") for col in morphology_aggregated.columns]
morphology_aggregated.rename(columns={"component_": "cluster_id"}, inplace=True)

morphology_aggregated

,component,area_mean,area_std,area_min,area_max,area_median,major_axis_length_mean,major_axis_length_std,major_axis_length_min,major_axis_length_max,...,boundary_irregularity_mean,boundary_irregularity_std,boundary_irregularity_min,boundary_irregularity_max,boundary_irregularity_median,nc_ratio_mean,nc_ratio_std,nc_ratio_min,nc_ratio_max,nc_ratio_median
0,34434_1_epithelial cells1,971.375000,318.488366,534.0,1413.0,951.5,41.787551,6.344545,31.413721,49.655243,...,3.782436,0.084233,3.655584,3.909831,3.783863,0.835448,0.056771,0.726772,0.898797,0.843821
1,34434_1_epithelial cells10,826.000000,358.949393,292.0,1595.0,752.0,40.786004,8.423006,23.397096,54.698792,...,3.878581,0.163454,3.641400,4.324321,3.886122,0.905092,0.477598,0.424908,2.807018,0.819196
2,34434_1_epithelial cells10001,786.166667,478.548813,241.0,1382.0,757.0,35.927149,11.203262,18.621426,46.681448,...,3.765168,0.102978,3.620237,3.911648,3.758217,0.702414,0.244669,0.421569,1.136929,0.647711
3,34434_1_epithelial cells10003,1195.600000,333.186584,743.0,1504.0,1260.0,51.801055,10.002341,39.213145,62.580616,...,3.941376,0.149990,3.787041,4.159087,3.953550,0.573629,0.183626,0.317969,0.788501,0.548413
4,34434_1_epithelial cells10007,744.561404,330.279320,104.0,1579.0,762.0,36.972646,9.704207,12.360754,52.931312,...,3.834246,0.179848,3.390165,4.284072,3.845899,0.776071,0.370054,0.211538,3.195946,0.771930
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9898,34434_1_epithelial cells9986,903.803030,380.803128,161.0,1914.0,916.5,40.106351,9.989119,14.721827,66.924745,...,3.820733,0.135720,3.434597,4.170895,3.812780,0.743187,0.530343,0.161491,4.805714,0.703723
9899,34434_1_epithelial cells999,599.379310,266.233556,62.0,1163.0,610.0,32.508544,8.544402,10.500690,50.603095,...,3.828370,0.143415,3.553670,4.288858,3.798366,0.914635,0.555922,0.323300,5.672897,0.815217
9900,34434_1_epithelial cells9990,538.800000,188.936674,281.0,786.0,563.0,31.103496,5.532347,20.595263,38.468956,...,3.859287,0.177555,3.652505,4.201265,3.812508,0.881631,0.452378,0.395018,1.878345,0.733791
9901,34434_1_epithelial cells9998,741.250000,343.486341,150.0,1416.0,649.5,35.328093,10.443240,18.882569,60.947698,...,3.726075,0.118116,3.569389,4.012372,3.701379,0.780571,0.448772,0.514706,2.413333,0.690387


In [20]:
morphology_aggregated["cluster_id"] = morphology_aggregated['component']

# read in polarity score and nuclear texture features

In [21]:
pixel_features = pd.read_csv("results/pixel_features.csv")
pixel_features

,label,geometric_centroid_x,geometric_centroid_y,intensity_centroid_x,intensity_centroid_y,polarity_score,haralick_contrast,haralick_correlation,haralick_energy,haralick_homogeneity,entropy,pcc_ck19_nak,intensity_ratio,inertia,lacunarity,fov
0,1,15.500000,13645.000000,13645.350628,15.192469,0.013717,49.740000,0.862273,0.642067,0.728770,2.254099,0.255596,7.267782,7.659819e+03,0.187118,fov0
1,2,16.582192,13321.746575,13320.957442,16.316379,0.005703,30.733766,0.968744,0.565918,0.626678,3.695876,0.252557,6.584596,1.390501e+05,0.514188,fov0
2,3,19.206897,12318.689655,12318.192045,19.172273,0.008600,133.152778,0.921746,0.584943,0.664681,2.870597,0.380406,3.875227,3.684614e+04,0.388584,fov0
3,4,21.146758,12301.972696,12302.124660,21.924030,0.002703,70.892720,0.978806,0.478306,0.590611,4.868242,0.207454,14.039366,1.235772e+06,0.952577,fov0
4,5,22.103448,10451.810345,10452.716244,22.846193,0.020198,62.480519,0.878593,0.613095,0.784326,2.795533,0.017809,17.954315,1.806678e+04,0.331055,fov0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1765837,187193,20233.932039,3381.912621,3381.892409,20233.223762,0.006879,117.783550,0.888166,0.526815,0.764971,3.813356,-0.458944,1.547525,1.500404e+05,0.144502,fov5
1765838,187194,20234.383562,3502.219178,3502.158456,20233.201671,0.008106,91.865385,0.948023,0.542841,0.655924,3.950150,-0.437838,1.530683,2.584634e+05,0.241357,fov5
1765839,187196,20235.663043,1547.059783,1546.824318,20233.914632,0.009588,35.840659,0.932528,0.469684,0.669925,4.651176,0.348424,0.260970,4.297418e+05,0.246110,fov5
1765840,187197,20236.182796,1620.827957,1620.976937,20234.802583,0.014927,47.473684,0.897505,0.524300,0.737715,3.746983,0.228187,0.073339,1.062711e+05,0.162754,fov5


In [22]:
pixel_features = pd.merge(pixel_features, 
                          adata_obs_component[["label", "fov", "component"]], 
                          on = ["label", "fov"], 
                          how="left")
pixel_features

,label,geometric_centroid_x,geometric_centroid_y,intensity_centroid_x,intensity_centroid_y,polarity_score,haralick_contrast,haralick_correlation,haralick_energy,haralick_homogeneity,entropy,pcc_ck19_nak,intensity_ratio,inertia,lacunarity,fov,component
0,1,15.500000,13645.000000,13645.350628,15.192469,0.013717,49.740000,0.862273,0.642067,0.728770,2.254099,0.255596,7.267782,7.659819e+03,0.187118,fov0,singleton
1,2,16.582192,13321.746575,13320.957442,16.316379,0.005703,30.733766,0.968744,0.565918,0.626678,3.695876,0.252557,6.584596,1.390501e+05,0.514188,fov0,singleton
2,3,19.206897,12318.689655,12318.192045,19.172273,0.008600,133.152778,0.921746,0.584943,0.664681,2.870597,0.380406,3.875227,3.684614e+04,0.388584,fov0,singleton
3,4,21.146758,12301.972696,12302.124660,21.924030,0.002703,70.892720,0.978806,0.478306,0.590611,4.868242,0.207454,14.039366,1.235772e+06,0.952577,fov0,singleton
4,5,22.103448,10451.810345,10452.716244,22.846193,0.020198,62.480519,0.878593,0.613095,0.784326,2.795533,0.017809,17.954315,1.806678e+04,0.331055,fov0,singleton
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1765837,187193,20233.932039,3381.912621,3381.892409,20233.223762,0.006879,117.783550,0.888166,0.526815,0.764971,3.813356,-0.458944,1.547525,1.500404e+05,0.144502,fov5,singleton
1765838,187194,20234.383562,3502.219178,3502.158456,20233.201671,0.008106,91.865385,0.948023,0.542841,0.655924,3.950150,-0.437838,1.530683,2.584634e+05,0.241357,fov5,singleton
1765839,187196,20235.663043,1547.059783,1546.824318,20233.914632,0.009588,35.840659,0.932528,0.469684,0.669925,4.651176,0.348424,0.260970,4.297418e+05,0.246110,fov5,singleton
1765840,187197,20236.182796,1620.827957,1620.976937,20234.802583,0.014927,47.473684,0.897505,0.524300,0.737715,3.746983,0.228187,0.073339,1.062711e+05,0.162754,fov5,singleton


In [23]:
# Group by tumor cluster and compute summary statistics
pixel_features_aggregated = pixel_features.groupby("component")[["polarity_score", 
                                                                 "haralick_contrast", 
                                                                 "haralick_correlation",
                                                                 "haralick_energy",
                                                                 "haralick_homogeneity",
                                                                 "entropy",
                                                                 "pcc_ck19_nak",
                                                                 "intensity_ratio",
                                                                 "inertia",
                                                                 "lacunarity"]].agg(
    ["mean", "std", "min", "max", "median"]
).reset_index()
pixel_features_aggregated

component polarity_score                          \
                                              mean       std           min   
0         34434_1_epithelial cells0       0.011341       NaN  1.134075e-02   
1         34434_1_epithelial cells1       0.002284  0.001224  1.818590e-04   
2        34434_1_epithelial cells10       0.003477  0.002784  7.038560e-04   
3       34434_1_epithelial cells100       0.010308  0.004195  7.341228e-03   
4      34434_1_epithelial cells1000       0.004568  0.002355  2.846496e-03   
...                             ...            ...       ...           ...   
27197  34434_1_epithelial cells9996       0.005352  0.005222  1.659439e-03   
27198  34434_1_epithelial cells9997       0.000275       NaN  2.746881e-04   
27199  34434_1_epithelial cells9998       0.004867  0.002986  5.301939e-04   
27200  34434_1_epithelial cells9999       0.002499  0.001704  3.778700e-04   
27201                     singleton       0.000959  0.001790  1.745328e-07   

                          haralick_contrast                          \
            max    median              mean         std         min   
0      0.011341  0.011341         89.750794         NaN   89.750794   
1      0.004207  0.002372        164.488472   84.994146   69.433532   
2      0.011330  0.002605        216.827346  133.368721   52.131250   
3      0.013274  0.010308        467.070019  138.994393  368.786141   
4      0.007252  0.003605        238.830199  124.566284  153.933884   
...         ...       ...               ...         ...         ...   
27197  0.009044  0.005352         72.962522   16.271562   61.456790   
27198  0.000275  0.000275         31.361749         NaN   31.361749   
27199  0.011180  0.004116        134.202435   81.423149   44.369613   
27200  0.009576  0.002127        209.235145  183.054489   10.983654   
27201  0.111006  0.000308        264.944923  282.372963    0.011080   

                     ...       inertia                              \
                max  ...          mean           std           min   
0         89.750794  ...  8.584203e+08           NaN  8.584203e+08   
1        341.237438  ...  2.783632e+09  2.267141e+09  8.643238e+08   
2        569.505556  ...  3.186010e+09  3.106308e+09  1.255852e+08   
3        565.353896  ...  5.639380e+08  4.253187e+08  2.631923e+08   
4        381.832924  ...  7.473038e+08  3.613945e+08  5.344733e+08   
...             ...  ...           ...           ...           ...   
27197     84.468254  ...  2.954809e+08  1.897465e+08  1.613099e+08   
27198     31.361749  ...  7.415917e+09           NaN  7.415917e+09   
27199    321.057143  ...  2.447603e+09  3.200581e+09  3.709513e+07   
27200    842.696053  ...  8.533227e+09  1.139952e+10  6.648619e+07   
27201  16996.701754  ...  2.121679e+08  3.660424e+08  0.000000e+00   

                                  lacunarity                                \
                max        median       mean       std       min       max   
0      8.584203e+08  8.584203e+08   2.097060       NaN  2.097060  2.097060   
1      7.842797e+09  1.990634e+09   0.840581  0.107858  0.652259  0.982567   
2      1.062167e+10  1.991983e+09   0.770083  0.279030  0.375603  1.660442   
3      8.646838e+08  5.639380e+08   0.642186  0.091672  0.577364  0.707007   
4      1.164578e+09  5.428601e+08   0.731730  0.041097  0.686473  0.766718   
...             ...           ...        ...       ...       ...       ...   
27197  4.296520e+08  2.954809e+08   0.530327  0.080221  0.473602  0.587051   
27198  7.415917e+09  7.415917e+09   0.290826       NaN  0.290826  0.290826   
27199  1.167805e+10  1.072573e+09   0.585452  0.187952  0.331225  0.880770   
27200  6.630550e+10  4.443468e+09   0.866306  0.459289  0.225566  2.280317   
27201  6.797874e+10  1.057300e+08   0.682512  0.305059  0.012161  7.494034   

                 
         median  
0      2.097060  
1      0.840544  
2      0.687982  
3      0.642186  
4      0.742000  
...         ...  
27197  0.530327  

In [24]:
# Flatten the column names
pixel_features_aggregated.columns = ["_".join(col).strip("_") for col in pixel_features_aggregated.columns]
pixel_features_aggregated.rename(columns={"component": "cluster_id"}, inplace=True)

pixel_features_aggregated

,cluster_id,polarity_score_mean,polarity_score_std,polarity_score_min,polarity_score_max,polarity_score_median,haralick_contrast_mean,haralick_contrast_std,haralick_contrast_min,haralick_contrast_max,...,inertia_mean,inertia_std,inertia_min,inertia_max,inertia_median,lacunarity_mean,lacunarity_std,lacunarity_min,lacunarity_max,lacunarity_median
0,34434_1_epithelial cells0,0.011341,NaN,1.134075e-02,0.011341,0.011341,89.750794,NaN,89.750794,89.750794,...,8.584203e+08,NaN,8.584203e+08,8.584203e+08,8.584203e+08,2.097060,NaN,2.097060,2.097060,2.097060
1,34434_1_epithelial cells1,0.002284,0.001224,1.818590e-04,0.004207,0.002372,164.488472,84.994146,69.433532,341.237438,...,2.783632e+09,2.267141e+09,8.643238e+08,7.842797e+09,1.990634e+09,0.840581,0.107858,0.652259,0.982567,0.840544
2,34434_1_epithelial cells10,0.003477,0.002784,7.038560e-04,0.011330,0.002605,216.827346,133.368721,52.131250,569.505556,...,3.186010e+09,3.106308e+09,1.255852e+08,1.062167e+10,1.991983e+09,0.770083,0.279030,0.375603,1.660442,0.687982
3,34434_1_epithelial cells100,0.010308,0.004195,7.341228e-03,0.013274,0.010308,467.070019,138.994393,368.786141,565.353896,...,5.639380e+08,4.253187e+08,2.631923e+08,8.646838e+08,5.639380e+08,0.642186,0.091672,0.577364,0.707007,0.642186
4,34434_1_epithelial cells1000,0.004568,0.002355,2.846496e-03,0.007252,0.003605,238.830199,124.566284,153.933884,381.832924,...,7.473038e+08,3.613945e+08,5.344733e+08,1.164578e+09,5.428601e+08,0.731730,0.041097,0.686473,0.766718,0.742000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27197,34434_1_epithelial cells9996,0.005352,0.005222,1.659439e-03,0.009044,0.005352,72.962522,16.271562,61.456790,84.468254,...,2.954809e+08,1.897465e+08,1.613099e+08,4.296520e+08,2.954809e+08,0.530327,0.080221,0.473602,0.587051,0.530327
27198,34434_1_epithelial cells9997,0.000275,NaN,2.746881e-04,0.000275,0.000275,31.361749,NaN,31.361749,31.361749,...,7.415917e+09,NaN,7.415917e+09,7.415917e+09,7.415917e+09,0.290826,NaN,0.290826,0.290826,0.290826
27199,34434_1_epithelial cells9998,0.004867,0.002986,5.301939e-04,0.011180,0.004116,134.202435,81.423149,44.369613,321.057143,...,2.447603e+09,3.200581e+09,3.709513e+07,1.167805e+10,1.072573e+09,0.585452,0.187952,0.331225,0.880770,0.559186
27200,34434_1_epithelial cells9999,0.002499,0.001704,3.778700e-04,0.009576,0.002127,209.235145,183.054489,10.983654,842.696053,...,8.533227e+09,1.139952e+10,6.648619e+07,6.630550e+10,4.443468e+09,0.866306,0.459289,0.225566,2.280317,0.744140


In [25]:
pixel_features_aggregated = pixel_features_aggregated[pixel_features_aggregated["cluster_id"].isin(cluster_of_interest)]

In [26]:
pixel_features_aggregated

,cluster_id,polarity_score_mean,polarity_score_std,polarity_score_min,polarity_score_max,polarity_score_median,haralick_contrast_mean,haralick_contrast_std,haralick_contrast_min,haralick_contrast_max,...,inertia_mean,inertia_std,inertia_min,inertia_max,inertia_median,lacunarity_mean,lacunarity_std,lacunarity_min,lacunarity_max,lacunarity_median
1,34434_1_epithelial cells1,0.002284,0.001224,0.000182,0.004207,0.002372,164.488472,84.994146,69.433532,341.237438,...,2.783632e+09,2.267141e+09,8.643238e+08,7.842797e+09,1.990634e+09,0.840581,0.107858,0.652259,0.982567,0.840544
2,34434_1_epithelial cells10,0.003477,0.002784,0.000704,0.011330,0.002605,216.827346,133.368721,52.131250,569.505556,...,3.186010e+09,3.106308e+09,1.255852e+08,1.062167e+10,1.991983e+09,0.770083,0.279030,0.375603,1.660442,0.687982
6,34434_1_epithelial cells10001,0.002776,0.001356,0.001029,0.004266,0.002851,92.310787,26.306534,65.439706,130.869031,...,2.132213e+09,2.008826e+09,2.008117e+08,4.607960e+09,1.483299e+09,0.684728,0.210181,0.402104,1.031914,0.656129
8,34434_1_epithelial cells10003,0.002019,0.001413,0.000573,0.004357,0.001803,158.200021,128.517670,11.996928,329.319342,...,4.172149e+09,3.036189e+09,9.714405e+08,8.193869e+09,2.928904e+09,0.902665,0.262298,0.554023,1.293813,0.887653
12,34434_1_epithelial cells10007,0.002793,0.002130,0.000492,0.014936,0.002130,401.689788,413.440775,31.895652,2643.287926,...,1.656450e+09,1.552187e+09,2.576290e+07,8.835966e+09,1.204129e+09,0.655620,0.224233,0.184361,1.126303,0.647681
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27186,34434_1_epithelial cells9986,0.002826,0.001893,0.000235,0.008530,0.002278,295.449141,327.903831,24.905240,2021.833333,...,4.046743e+09,3.638108e+09,4.317987e+07,2.141986e+10,3.716103e+09,0.683150,0.278348,0.230068,1.682554,0.637966
27190,34434_1_epithelial cells999,0.004186,0.002296,0.000050,0.014283,0.003771,437.120322,326.633712,41.665152,1952.933333,...,1.425010e+09,1.384188e+09,8.537713e+06,6.808993e+09,9.639917e+08,0.614231,0.204954,0.190072,1.329815,0.596552
27191,34434_1_epithelial cells9990,0.003812,0.002270,0.001446,0.008775,0.002781,881.963536,445.852509,459.432234,1675.011442,...,5.521096e+08,3.238612e+08,1.657130e+08,1.148172e+09,5.902838e+08,0.420603,0.176887,0.215463,0.733345,0.345401
27199,34434_1_epithelial cells9998,0.004867,0.002986,0.000530,0.011180,0.004116,134.202435,81.423149,44.369613,321.057143,...,2.447603e+09,3.200581e+09,3.709513e+07,1.167805e+10,1.072573e+09,0.585452,0.187952,0.331225,0.880770,0.559186


# organize combined_cluster_features and hole_features data frames

In [27]:
hole_aggregated = hole_features.groupby("cluster_id").agg({
    "hole_area": ["sum", "mean", "count"],  # Total, mean, count of holes
    "hole_perimeter": ["sum", "mean"],
    "hole_eccentricity": "mean",  # Average eccentricity of holes
    "hole_circularity": "mean", # Average circularity of holes
    "hole_fractal_dimension": "mean",
    "hole_orientation": "mean"
}).reset_index()

# Flatten MultiIndex Columns for Aggregated Hole Features
hole_aggregated.columns = ["_".join(col).strip("_") if isinstance(col, tuple) else col for col in hole_aggregated.columns]

In [28]:
# Add weights to the aggregation by subcluster area
def weighted_mean(df, value_col, weight_col):
    """Calculate the weighted mean of a column."""
    return (df[value_col] * df[weight_col]).sum() / df[weight_col].sum()

# Perform weighted aggregation for combined features
weighted_aggregated = combined_cluster_features.groupby("cluster_id").apply(
    lambda group: pd.Series({
        "total_area": group["area"].sum(),
        "mean_perimeter": weighted_mean(group, "perimeter", "area"),
        "mean_eccentricity": weighted_mean(group, "eccentricity", "area"),
        "mean_convexity": weighted_mean(group, "convexity", "area"),
        "mean_fractal_dimension": weighted_mean(group, "fractal_dimension", "area"),
        "mean_elongation": weighted_mean(group, "elongation", "area"),
        "mean_circularity": weighted_mean(group, "circularity", "area"),
        "mean_orientation": weighted_mean(group, "orientation", "area"),
        "mean_radial_deviation": weighted_mean(group, "mean_radial_deviation", "area"),
        "layer_density_variance": weighted_mean(group, "layer_density_variance", "area"),
        "layer_thickness_variance": weighted_mean(group, "layer_thickness_variance", "area"),
        "spatial_entropy": weighted_mean(group, "spatial_entropy", "area"),
    })
).reset_index()

weighted_aggregated

/var/folders/kg/btvhyzjs5fb7f_57bfqf24tr0000gn/T/ipykernel_24186/660536668.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weighted_aggregated = combined_cluster_features.groupby("cluster_id").apply(


,cluster_id,total_area,mean_perimeter,mean_eccentricity,mean_convexity,mean_fractal_dimension,mean_elongation,mean_circularity,mean_orientation,mean_radial_deviation,layer_density_variance,layer_thickness_variance,spatial_entropy
0,34434_1_epithelial cells1,7771.0,685.511760,0.862228,0.716289,0.704377,1.974257,0.207805,0.615311,4580.417186,18.0,2.154116e-26,0.376770
1,34434_1_epithelial cells10,15694.0,420.921099,0.929854,0.833334,0.679609,3.153441,0.374048,-0.765631,4620.498434,144.5,2.194973e-26,0.206192
2,34434_1_epithelial cells10001,4717.0,297.774342,0.771431,0.806679,0.676470,1.861158,0.474585,-0.086180,23949.863142,8.0,1.263748e-24,0.450561
3,34434_1_epithelial cells10003,5978.0,495.587878,0.930155,0.753906,0.690998,2.723550,0.305861,0.012000,19894.671807,4.5,3.676358e-25,0.500402
4,34434_1_epithelial cells10007,42440.0,2588.757678,0.812255,0.391994,0.722516,1.722607,0.094827,1.504981,21747.952618,1512.5,4.514129e-25,0.088320
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9897,34434_1_epithelial cells9986,59651.0,3116.740204,0.894895,0.377485,0.715517,2.246641,0.091015,0.883882,14621.229036,2048.0,3.376933e-25,0.078516
9898,34434_1_epithelial cells999,95034.0,477.276313,0.816237,0.817430,0.660884,2.039593,0.521779,-0.358428,14801.251489,10224.5,2.589692e-25,0.041195
9899,34434_1_epithelial cells9990,5388.0,365.705627,0.671172,0.809488,0.654149,1.348978,0.506261,1.038067,29776.677309,32.0,1.838179e-24,0.325083
9900,34434_1_epithelial cells9998,11860.0,232.674632,0.751302,0.857367,0.666806,1.738720,0.589736,0.176811,24191.693556,98.0,3.434400e-25,0.233792


# Merge merged_df and weighted_aggregated on "cluster_id"

In [29]:
merged_with_weighted = merged_df.merge(weighted_aggregated, on="cluster_id", how="outer")

# Merge the result with hole_aggregated on "cluster_id"
final_combined_df = merged_with_weighted#.merge(hole_aggregated, on="cluster_id", how="outer")

# Fill missing values with 0
final_combined_df.fillna(0, inplace=True)

# Optional: Inspect the final DataFrame
print(final_combined_df.head())

                      cluster_id  total_cells  proliferating_cells  \
0      34434_1_epithelial cells1            8                    5   
1     34434_1_epithelial cells10           19                    8   
2  34434_1_epithelial cells10001            6                    2   
3  34434_1_epithelial cells10003            5                    0   
4  34434_1_epithelial cells10007           57                    0   

   proliferating_index   fov  max_thickness  mean_thickness  median_thickness  \
0            62.500000  fov0      39.395431        9.428391               4.0   
1            42.105263  fov0      51.224994        8.507611               4.0   
2            33.333333  fov2      36.221541       10.903195               4.0   
3             0.000000  fov2      44.721360       14.564069               4.0   
4             0.000000  fov2      71.217975       13.064024               4.0   

   std_thickness  total_area  ...  mean_eccentricity  mean_convexity  \
0       9.599122    

In [30]:
# Remove specific columns from the DataFrame
final_combined_df = final_combined_df.drop(columns=["proliferating_cells", "fov"])
# Optional: Verify the result
print(final_combined_df.head())

                      cluster_id  total_cells  proliferating_index  \
0      34434_1_epithelial cells1            8            62.500000   
1     34434_1_epithelial cells10           19            42.105263   
2  34434_1_epithelial cells10001            6            33.333333   
3  34434_1_epithelial cells10003            5             0.000000   
4  34434_1_epithelial cells10007           57             0.000000   

   max_thickness  mean_thickness  median_thickness  std_thickness  total_area  \
0      39.395431        9.428391               4.0       9.599122      7771.0   
1      51.224994        8.507611               4.0      10.366476     15694.0   
2      36.221541       10.903195               4.0      10.027408      4717.0   
3      44.721360       14.564069               4.0      12.916919      5978.0   
4      71.217975       13.064024               4.0      14.585605     42440.0   

   mean_perimeter  mean_eccentricity  mean_convexity  mean_fractal_dimension  \
0      685.5

# Merge Aggregated Morphology Features with final_combined_df

In [31]:
final_combined_df = final_combined_df.merge(
    morphology_aggregated, on="cluster_id", how="left"
)
# Check merged dataset
final_combined_df.head()

,cluster_id,total_cells,proliferating_index,max_thickness,mean_thickness,median_thickness,std_thickness,total_area,mean_perimeter,mean_eccentricity,...,boundary_irregularity_mean,boundary_irregularity_std,boundary_irregularity_min,boundary_irregularity_max,boundary_irregularity_median,nc_ratio_mean,nc_ratio_std,nc_ratio_min,nc_ratio_max,nc_ratio_median
0,34434_1_epithelial cells1,8,62.500000,39.395431,9.428391,4.0,9.599122,7771.0,685.511760,0.862228,...,3.782436,0.084233,3.655584,3.909831,3.783863,0.835448,0.056771,0.726772,0.898797,0.843821
1,34434_1_epithelial cells10,19,42.105263,51.224994,8.507611,4.0,10.366476,15694.0,420.921099,0.929854,...,3.878581,0.163454,3.641400,4.324321,3.886122,0.905092,0.477598,0.424908,2.807018,0.819196
2,34434_1_epithelial cells10001,6,33.333333,36.221541,10.903195,4.0,10.027408,4717.0,297.774342,0.771431,...,3.765168,0.102978,3.620237,3.911648,3.758217,0.702414,0.244669,0.421569,1.136929,0.647711
3,34434_1_epithelial cells10003,5,0.000000,44.721360,14.564069,4.0,12.916919,5978.0,495.587878,0.930155,...,3.941376,0.149990,3.787041,4.159087,3.953550,0.573629,0.183626,0.317969,0.788501,0.548413
4,34434_1_epithelial cells10007,57,0.000000,71.217975,13.064024,4.0,14.585605,42440.0,2588.757678,0.812255,...,3.834246,0.179848,3.390165,4.284072,3.845899,0.776071,0.370054,0.211538,3.195946,0.771930


In [32]:
final_combined_df = final_combined_df.merge(
    pixel_features_aggregated, on="cluster_id", how="left"
)
# Check merged dataset
final_combined_df.head()

,cluster_id,total_cells,proliferating_index,max_thickness,mean_thickness,median_thickness,std_thickness,total_area,mean_perimeter,mean_eccentricity,...,inertia_mean,inertia_std,inertia_min,inertia_max,inertia_median,lacunarity_mean,lacunarity_std,lacunarity_min,lacunarity_max,lacunarity_median
0,34434_1_epithelial cells1,8,62.500000,39.395431,9.428391,4.0,9.599122,7771.0,685.511760,0.862228,...,2.783632e+09,2.267141e+09,8.643238e+08,7.842797e+09,1.990634e+09,0.840581,0.107858,0.652259,0.982567,0.840544
1,34434_1_epithelial cells10,19,42.105263,51.224994,8.507611,4.0,10.366476,15694.0,420.921099,0.929854,...,3.186010e+09,3.106308e+09,1.255852e+08,1.062167e+10,1.991983e+09,0.770083,0.279030,0.375603,1.660442,0.687982
2,34434_1_epithelial cells10001,6,33.333333,36.221541,10.903195,4.0,10.027408,4717.0,297.774342,0.771431,...,2.132213e+09,2.008826e+09,2.008117e+08,4.607960e+09,1.483299e+09,0.684728,0.210181,0.402104,1.031914,0.656129
3,34434_1_epithelial cells10003,5,0.000000,44.721360,14.564069,4.0,12.916919,5978.0,495.587878,0.930155,...,4.172149e+09,3.036189e+09,9.714405e+08,8.193869e+09,2.928904e+09,0.902665,0.262298,0.554023,1.293813,0.887653
4,34434_1_epithelial cells10007,57,0.000000,71.217975,13.064024,4.0,14.585605,42440.0,2588.757678,0.812255,...,1.656450e+09,1.552187e+09,2.576290e+07,8.835966e+09,1.204129e+09,0.655620,0.224233,0.184361,1.126303,0.647681


# Load tumour cluster ourter border cell type proportion df

In [33]:
phenotype_proportion_df = pd.read_pickle("tumour_cluster_outer_border_phenotype_proportion.pkl")
phenotype_proportion_df

,34434_1_epithelial cells26135,34434_1_epithelial cells6442,34434_1_epithelial cells12370,34434_1_epithelial cells5871,34434_1_epithelial cells9810,34434_1_epithelial cells17936,34434_1_epithelial cells5645,34434_1_epithelial cells6882,34434_1_epithelial cells17873,34434_1_epithelial cells4504,...,34434_1_epithelial cells7274,34434_1_epithelial cells12346,34434_1_epithelial cells7291,34434_1_epithelial cells1431,34434_1_epithelial cells5709,34434_1_epithelial cells2807,34434_1_epithelial cells2773,34434_1_epithelial cells12384,34434_1_epithelial cells18425,34434_1_epithelial cells18386
phenotype,,,,,,,,,,,,,,,,,,,,,
B cells,0.000977,0.005931,0.001773,0.006486,0.005763,0.042898,0.015065,0.003185,0.110061,0.021028,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.023810,0.000000,0.000000,0.029412,0.000000
NK cells,0.004494,0.016145,0.013681,0.012432,0.000699,0.005720,0.010713,0.013447,0.007186,0.175234,...,0.047619,0.111111,0.000000,0.127660,0.055556,0.309524,0.000000,0.307692,0.117647,0.000000
T cells,0.003322,0.032619,0.017482,0.022432,0.006811,0.043851,0.057918,0.014508,0.031014,0.024700,...,0.047619,0.037037,0.000000,0.000000,0.000000,0.047619,0.000000,0.025641,0.029412,0.000000
Unknown,0.002345,0.039703,0.008614,0.024054,0.000175,0.000000,0.019083,0.033263,0.009455,0.000668,...,0.023810,0.000000,0.035088,0.000000,0.018519,0.000000,0.090909,0.000000,0.029412,0.000000
endothelial cells,0.007425,0.065404,0.054218,0.031351,0.018861,0.020019,0.050218,0.057679,0.031014,0.029039,...,0.214286,0.037037,0.263158,0.127660,0.037037,0.095238,0.045455,0.000000,0.029412,0.000000
epithelial cells,0.893904,0.655519,0.785660,0.770000,0.464548,0.707658,0.701373,0.736730,0.663011,0.600134,...,0.238095,0.259259,0.210526,0.106383,0.296296,0.166667,0.318182,0.230769,0.147059,0.104167
fibroblasts,0.002735,0.038715,0.017735,0.041351,0.000000,0.000318,0.058252,0.040694,0.001135,0.018358,...,0.190476,0.370370,0.070175,0.042553,0.000000,0.261905,0.181818,0.102564,0.000000,0.000000
other epithelial cells,0.083626,0.023723,0.038257,0.017568,0.133426,0.100095,0.004352,0.010970,0.107791,0.110814,...,0.142857,0.074074,0.228070,0.531915,0.537037,0.047619,0.090909,0.179487,0.500000,0.895833
plasma cells,0.000195,0.033443,0.007094,0.005405,0.369717,0.073085,0.021091,0.005662,0.021558,0.005674,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.023810,0.000000,0.000000,0.000000,0.000000


In [34]:
# Transpose phenotype_proportion_df
phenotype_proportion_df_T = phenotype_proportion_df.T  # Transpose
phenotype_proportion_df_T.index.name = "cluster_id"  # Rename index
phenotype_proportion_df_T.reset_index(inplace=True)  # Convert index to column
phenotype_proportion_df_T

phenotype,cluster_id,B cells,NK cells,T cells,Unknown,endothelial cells,epithelial cells,fibroblasts,other epithelial cells,plasma cells,smooth muscle cells,stromal cells
0,34434_1_epithelial cells26135,0.000977,0.004494,0.003322,0.002345,0.007425,0.893904,0.002735,0.083626,0.000195,0.000977,0.000000
1,34434_1_epithelial cells6442,0.005931,0.016145,0.032619,0.039703,0.065404,0.655519,0.038715,0.023723,0.033443,0.031796,0.057002
2,34434_1_epithelial cells12370,0.001773,0.013681,0.017482,0.008614,0.054218,0.785660,0.017735,0.038257,0.007094,0.028629,0.026856
3,34434_1_epithelial cells5871,0.006486,0.012432,0.022432,0.024054,0.031351,0.770000,0.041351,0.017568,0.005405,0.034054,0.034865
4,34434_1_epithelial cells9810,0.005763,0.000699,0.006811,0.000175,0.018861,0.464548,0.000000,0.133426,0.369717,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
9898,34434_1_epithelial cells2807,0.023810,0.309524,0.047619,0.000000,0.095238,0.166667,0.261905,0.047619,0.023810,0.023810,0.000000
9899,34434_1_epithelial cells2773,0.000000,0.000000,0.000000,0.090909,0.045455,0.318182,0.181818,0.090909,0.000000,0.136364,0.136364
9900,34434_1_epithelial cells12384,0.000000,0.307692,0.025641,0.000000,0.000000,0.230769,0.102564,0.179487,0.000000,0.000000,0.153846
9901,34434_1_epithelial cells18425,0.029412,0.117647,0.029412,0.029412,0.029412,0.147059,0.000000,0.500000,0.000000,0.000000,0.117647


In [35]:
# Merge with final_combined_df
final_combined_df = final_combined_df.merge(phenotype_proportion_df_T, on="cluster_id", how="left")

In [36]:
print(final_combined_df.head())

                      cluster_id  total_cells  proliferating_index  \
0      34434_1_epithelial cells1            8            62.500000   
1     34434_1_epithelial cells10           19            42.105263   
2  34434_1_epithelial cells10001            6            33.333333   
3  34434_1_epithelial cells10003            5             0.000000   
4  34434_1_epithelial cells10007           57             0.000000   

   max_thickness  mean_thickness  median_thickness  std_thickness  total_area  \
0      39.395431        9.428391               4.0       9.599122      7771.0   
1      51.224994        8.507611               4.0      10.366476     15694.0   
2      36.221541       10.903195               4.0      10.027408      4717.0   
3      44.721360       14.564069               4.0      12.916919      5978.0   
4      71.217975       13.064024               4.0      14.585605     42440.0   

   mean_perimeter  mean_eccentricity  ...  NK cells   T cells   Unknown  \
0      685.511760

In [37]:
final_combined_df.to_pickle("tumour_cluster_final_combined_df.pkl")